# What each strategy would make of the dataset

This notebook runs every feature selection strategy written in `src/selector/strategies/` over the features computed on disk and reads the results side by side.
It is a prediction of the resulting dataset, in order to choose the most appropriate approach to the problem.
Consider `notebooks/coverage.ipynb` as a way to perform a qualitative assessment of the strategies, in order to better understand what they would produce if run on the real data.

## Setup

In [1]:
"""Import the prediction and the tables that read it."""

from sampling.stats import catalogue
from visualization.dataset import progress
from visualization.dataset.plots import reach
from visualization.dataset.tables import coverage, overall, size, tiling

## The ODE dataset

How much there is to choose from, before any strategy is asked of it. Every
number here is read off the measurement itself, so it is the same whichever
strategy the rest of the notebook goes on to compare.

### The ODE dataset that was measured

Read straight off the coverage measurement, so it is the same whichever strategy
the rest of the notebook compares.

- **Features** is how many the coverage stage measured.
- **Feature classes** is how many kinds the catalogue files them under.
- **Features dropped as points** is how many the catalogue gives no extent, so
  there was no ground to crop an observation to and nothing was downloaded.
- **Ground** is what their bounding boxes hold between them.

In [8]:
"""Read the catalogue index, which every table in this section is drawn from."""

measured = catalogue.read()
overall.measured(measured)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

### Global instrument coverage

One row per instrument, over the whole measured record and before any strategy.

- **Features reached** is how many features it took any observation of.
- **Observations** is how many it took of them altogether.
- **Ground reached** and **Share of the ground** are how much of the features'
  ground it covered, counting a cell once however often it was revisited.
- **First look** and **Last look** bound the record a window may be chosen from.

In [10]:
"""Tabulate what each instrument holds of the measured features."""

overall.instruments(measured)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

### Median share of a feature reached, against how big it is

Whether an instrument stops covering a feature once the feature gets big, which
is what decides how wide a tile can usefully be cut.

The horizontal axis is the ground a feature's bounding box holds, on a log scale.
The vertical axis is the share of that ground the instrument reached over its
whole record. Every feature falls into one of 18 log-spaced size bands, and one
point is drawn per band per instrument at the **median** of the features in it,
skipping any band holding fewer than five. It is a median rather than a mean so
that a handful of fully covered outliers cannot lift a band.

In [11]:
"""Draw how much of a feature each instrument reached against its size."""

reach.against_size(measured)

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04L\x00\x00\x01\xcc\x08\x06\x00\x00\x00\xcf\x89=\xc…

### What each feature class holds

The landforms the catalogue offers, most numerous first.

- **Features measured** is how many of that class the coverage stage measured.
- **Mean feature size** is the ground the average one of them holds, which is
  what decides whether a strategy's tile width cuts it up at all.

In [ ]:
"""Tabulate how many features each class holds and how big they are."""

overall.held(measured)

## Sweep the features

If a new strategy has not been computed, it could take several minutes to fill in the missing stats.

In [2]:
"""Read what every strategy makes of every tile of every feature."""

read = progress.read()

## Tiling

Each strategy splits the features it is given into tiles by its own rules.
This section focuses on what each tile is ideally expected to cover.
It's important to note that the tile width is an upper bound, not a fixed size: the tiling algorithm will pick the fewest even divisions of the features that are smaller than the cap, so a tile may be smaller than the width asked.

### How big a tile is

A window is searched over one tile at a time, so the tile is the unit everything
below is measured on.

- **Tile width asked** is `tile_km`, written in the strategy's own YAML file. It
  is a cap, not a size: `tiling.split` picks the fewest even divisions of the
  feature's grid that stay under it, so the real side is at most this wide.
- **Tiles created** counts every tile the split produced across all features.
- **Mean tile width**, with its **Narrowest** and **Widest**, is the side of a
  square holding the feature ground one tile covers. A tile counts only the cells
  inside the feature outline, so one clipping the rim of a crater holds a
  fraction of its own footprint, and this always reads below the width asked.

In [3]:
"""Tabulate how big a tile each strategy cuts its features into."""

tiling.sizes(read)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

### What it takes to fill a tile

How much of an instrument a whole tile of the asked width would swallow. It is
worked out from the tile width asked and the median ground one pixel covers, not
from what any observation happened to land, so two strategies cutting to the same
width read the same here.

**SHARAD is counted as a line, not as a picture.** Its two counts are **traces**,
positions along the ground track laid down one every 460 m. They say how many
times a single pass sounds the ground while crossing a tile. They say nothing
about how deep each sounding looks, which is why a sounder is credited with no
count for filling a tile at all.

Each trace is itself a column of depth samples. SHARAD transmits over a 10 MHz
bandwidth centred on 20 MHz, giving a range resolution of about 15 m in free
space and about 8.4 m in water ice, whose refractive index is near 1.78.
Sounding to roughly the 1 km of ice SHARAD is credited with, one trace therefore
carries on the order of 120 depth samples of real subsurface, or about 67 if the
free space figure is used. So a pass across a tile leaves the traces counted
below, each roughly a hundred samples deep, rather than that many pixels.

In [ ]:
"""Tabulate how much of each instrument it takes to cover a whole tile."""

tiling.filling(read)

### What each instrument lands on a tile

Whether an instrument brings a tile enough to be worth asking it for anything.

- **Mean observations offered** counts the observations of that instrument whose
  footprint lands on a tile at all, over the whole twenty year record and before
  any window is chosen. It is not what a window keeps. A strategy whose tile
  width leaves each feature as a single tile puts every observation of that
  feature onto that one tile, which is why it reads in the hundreds there and in
  the tens once the features are really cut into 100 km tiles.
- **Mean pixels landed** is the pixels the window's own observations leave on a
  tile, added over them, counted as traces for SHARAD.
- **Mean pixels landed per observation** divides those pixels by the
  observations that landed them, so it is what one look at a tile is worth.
- **Pixels asked** is `admits` from the strategy: the pixels an instrument has to
  land on a whole tile before the observation counts as a look at the tile
  rather than a clip of its edge. An observation below it is refused outright.

In [4]:
"""Tabulate what each instrument lands on a tile and what is asked of it."""

tiling.landed(read)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

## Coverage

Now that the initial tiling situation has been described, we can look at what each strategy would have kept and thrown away.

### How much of a tile each instrument reaches

What the chosen window actually leaves on the ground, counting a cell once
however often it was revisited. Only tiles that earned a window are counted, and
an instrument absent from one counts as nothing there rather than being left out.

- **Mean coverage inside a tile** is the share of the tile the instrument
  reaches, averaged over every kept tile.
- **Least** is the worst tile, which should sit at or just above the floor the
  strategy asks of that instrument.
- The **Overlap** row is the share of a tile that *every* instrument reaches at
  once, which is what the feature would look like if all three had to be
  registered together.

In [5]:
"""Tabulate how much of a tile each instrument reaches and how much they share."""

coverage.reached(read)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

### How long a window runs

- **Mean window** and **Longest** are how far apart the earliest and latest
  observations a kept tile holds are, capped by the strategy's `span_days`.
- **Time Window Score** is the geometric mean of the shares the window's own
  constraints reach, so as the strategies are written now it is exactly
  `sqrt(CTX share of the tile x CRISM share of the tile)`. SHARAD is `timeless`,
  so it is asked of the whole record rather than of the window and never enters
  the score. The days a window runs are priced against the ground it reaches
  while the search is choosing the window, but that charge is not carried into
  the number reported here.

The score reads about the same under every strategy, and that is the honest
answer rather than a bug. All four strategies ask for the same `constraints`, so
all four share the same floor, `sqrt(0.50 x 0.30)` = 38.7%, and the score is only
ever measured on tiles that had already cleared it. Neither `tile_km` nor
`span_days` enters the score at all. What separates the strategies is how many
tiles clear that floor, which is the next section.

In [6]:
"""Tabulate how long a window runs and how far it reaches."""

coverage.windows(read)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

## Final dataset size

What each strategy would hand over: the tiles it keeps of the tiles it
searched, how many of them hold more than one instrument, and how many of
them two instruments really overlap on rather than merely sharing a tile.

### The dataset each strategy would leave

- **Tiles searched** is every tile holding anything measurable.
- **Tiles kept** is those that earned a window worth keeping, so the tiles the
  dataset would actually be built from.
- **Share kept** is the second over the first, and is the number that really
  separates one strategy from another.

In [7]:
"""Tabulate the dataset each strategy would leave behind."""

size.final(read)

HTML(value='<div style="font-family: sans-serif; font-size: 13px;">\n          <div style="font-weight: 600; m…

## What each strategy makes of each feature class

The same three questions asked of every landform in turn: how many survive, how
much of one the dataset would hold, and how long it would have to watch. A
feature counts as **selected** when at least one of its tiles earned a window
worth keeping, however many of its other tiles were refused.

The two tables that average over features average over the selected ones only,
since a feature a strategy refused outright would otherwise drag its class down
to nothing. A class no strategy selected reads `none` under it.

### How many features of each class each strategy would select

Read this against **Features measured** in *What each feature class holds*
above. It is the table that says which kinds of landform survive a strategy and
which it wipes out entirely.

In [ ]:
"""Tabulate how many features of each class each strategy would select."""

overall.selected(measured, read)

### How much of a selected feature each strategy would hold

The share of a feature the dataset would actually carry. A feature is cut into
tiles, and some strategies leave it as a single tile while others split it into
many, so the share is read as the **mean over every tile the feature was split
into** of the ground any instrument reaches on that tile. A tile that earned no
window counts as nothing, which is what makes a strategy that keeps only a
corner of a feature read low here even when it selected the feature.

In [ ]:
"""Tabulate how much of a feature of each class each strategy would hold."""

overall.covered(measured, read)

### How long a window runs on a selected feature

Averaged first over the kept tiles of one feature, then over the selected
features of the class, so a feature cut into many tiles weighs the same as one
left whole. It says which landforms a strategy has to watch longest before it
has gathered what the strategy asks for.

In [ ]:
"""Tabulate how long a window runs on a feature of each class."""

overall.lasting(measured, read)